In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr
from pathlib import Path
import pandas_datareader.data as web
import datetime

ROOT = Path(r"C:\Users\navon\OneDrive\Desktop\CrossRank")

# --- reload base features/targets ---
base = pd.read_csv(ROOT / "data/processed/uk_etf_features_targets.csv", parse_dates=["date"])
base = base.sort_values(["ticker", "date"]).reset_index(drop=True)

FEATURE_COLS = ["mom_1m","mom_3m","mom_6m","mom_12m","realized_vol","max_drawdown_12m"]
MACRO_COLS = ["uk_rate", "uk_gilt_10y", "gbpusd", "vix"]
FEATURE_COLS_V2 = FEATURE_COLS + MACRO_COLS

# --- 3-month forward target (same as V3 in notebook 04) ---
base["fwd_ret_3m"] = (
    base.groupby("ticker")["ret"]
    .transform(lambda s: (1 + s.shift(-1)) * (1 + s.shift(-2)) * (1 + s.shift(-3)) - 1)
)
base["xs_excess_ret_3m"] = base["fwd_ret_3m"] - base.groupby("date")["fwd_ret_3m"].transform("mean")

for col in FEATURE_COLS:
    base[col] = base.groupby("ticker")[col].shift(1)

# --- reload macro data ---
start = datetime.datetime(2008, 1, 1)
end = datetime.datetime(2026, 10, 31)
series_codes = {"uk_rate": "IUDZLS9", "uk_gilt_10y": "IRLTLT01GBM156N", "gbpusd": "DEXUSUK", "vix": "VIXCLS"}
macro_raw = {name: web.DataReader(code, "fred", start, end) for name, code in series_codes.items()}
macro_monthly = {name: s[s.columns[0]].resample("ME").last().ffill() for name, s in macro_raw.items()}
macro_df = pd.DataFrame(macro_monthly).reset_index().rename(columns={"DATE": "date"})
macro_lagged = macro_df.copy()
macro_lagged[MACRO_COLS] = macro_lagged[MACRO_COLS].shift(1)

data_v3 = base.merge(macro_lagged, on="date", how="left")
required_complete = FEATURE_COLS + ["uk_gilt_10y","gbpusd","vix","xs_excess_ret_3m"]
data_v3 = data_v3.dropna(subset=required_complete).reset_index(drop=True)

# --- rebuild fold schedule ---
TRAIN_MONTHS, TEST_MONTHS = 36, 3
all_dates = sorted(data_v3["date"].unique())
folds = []
i = TRAIN_MONTHS
while i + TEST_MONTHS <= len(all_dates):
    train_dates, test_dates = all_dates[i-TRAIN_MONTHS:i], all_dates[i:i+TEST_MONTHS]
    folds.append({"train_start": train_dates[0], "train_end": train_dates[-1],
                   "test_start": test_dates[0], "test_end": test_dates[-1]})
    i += TEST_MONTHS

print(f"data_v3 rows: {len(data_v3):,}")
print(f"Folds: {len(folds)}")
print(f"Date range: {data_v3['date'].min().date()} to {data_v3['date'].max().date()}")

data_v3 rows: 36,930
Folds: 53
Date range: 2010-04-30 to 2026-07-31


In [2]:
oos_predictions = []

for f in folds:
    train_mask = (data_v3["date"] >= f["train_start"]) & (data_v3["date"] <= f["train_end"])
    test_mask = (data_v3["date"] >= f["test_start"]) & (data_v3["date"] <= f["test_end"])
    train, test = data_v3[train_mask], data_v3[test_mask]
    if len(train) < 50 or len(test) < 5:
        continue

    X_train_raw, y_train = train[FEATURE_COLS_V2], train["xs_excess_ret_3m"]
    X_test_raw = test[FEATURE_COLS_V2]
    X_train_filled, X_test_filled = X_train_raw.fillna(0), X_test_raw.fillna(0)

    lgb_model = lgb.LGBMRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                                   min_child_samples=20, random_state=42, verbose=-1)
    lgb_model.fit(X_train_raw, y_train)
    lgb_pred = lgb_model.predict(X_test_raw)

    rf_model = RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=20,
                                      random_state=42, n_jobs=-1)
    rf_model.fit(X_train_filled, y_train)
    rf_pred = rf_model.predict(X_test_filled)

    pred_df = pd.DataFrame({"LightGBM": lgb_pred, "RF": rf_pred})
    ranked = pred_df.rank(pct=True)
    blended_pred = ranked.mean(axis=1)

    test_out = test[["ticker","date","ret","fwd_ret_3m","xs_excess_ret_3m"]].copy()
    test_out["pred_score"] = blended_pred.values
    oos_predictions.append(test_out)

oos_df = pd.concat(oos_predictions, ignore_index=True)
print(f"Total OOS prediction rows: {len(oos_df):,}")
print(f"Date range: {oos_df['date'].min().date()} to {oos_df['date'].max().date()}")
print(f"Unique test months: {oos_df['date'].nunique()}")
oos_df.head(10)

Total OOS prediction rows: 35,210
Date range: 2013-04-30 to 2026-06-30
Unique test months: 159


,ticker,date,ret,fwd_ret_3m,xs_excess_ret_3m,pred_score
0,BRIC,2013-04-30,-0.024609,-0.061443,-0.055732,0.034591
1,BRIC,2013-05-31,-0.007421,-0.081638,-0.037245,0.066038
2,BRIC,2013-06-30,-0.080742,0.042290,-0.001790,0.333333
3,DJMC,2013-04-30,0.034014,0.066941,0.072651,0.078616
4,DJMC,2013-05-31,0.024342,0.025851,0.070244,0.352201
5,DJMC,2013-06-30,-0.044637,0.100336,0.056256,0.512579
6,DJSC,2013-04-30,0.025931,0.052803,0.058513,0.072327
7,DJSC,2013-05-31,0.029279,0.001459,0.045853,0.323899
8,DJSC,2013-06-30,-0.062241,0.094374,0.050294,0.421384
9,DL2P,2013-04-30,0.027193,0.103898,0.109608,0.169811


In [3]:
out_path = ROOT / "data/processed/oos_predictions.csv"
oos_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Size: {round(out_path.stat().st_size/1e6, 2)} MB")

Saved: C:\Users\navon\OneDrive\Desktop\CrossRank\data\processed\oos_predictions.csv
Size: 3.41 MB


In [4]:
# identify the actual rebalance dates — every 3rd test month, non-overlapping
all_test_months = sorted(oos_df["date"].unique())
rebalance_dates = all_test_months[::3]  # every 3rd month = non-overlapping with 3m holding

print(f"Total test months: {len(all_test_months)}")
print(f"Rebalance dates (every 3rd): {len(rebalance_dates)}")

rebalance_df = oos_df[oos_df["date"].isin(rebalance_dates)].copy()
print(f"Rebalance rows: {len(rebalance_df):,}")

def build_portfolio_returns(df, method, param):
    """
    method: 'top_pct' (param=fraction, e.g. 0.10), 'top_n' (param=count),
            'score_weighted' (param=fraction to include, weighted by score)
    Returns a Series of portfolio returns, one per rebalance date.
    """
    portfolio_rets = []
    dates_used = []
    for date, g in df.groupby("date"):
        g = g.sort_values("pred_score", ascending=False)
        if method == "top_pct":
            n = max(1, int(len(g) * param))
            selected = g.head(n)
            port_ret = selected["fwd_ret_3m"].mean()
        elif method == "top_n":
            selected = g.head(param)
            port_ret = selected["fwd_ret_3m"].mean()
        elif method == "score_weighted":
            n = max(1, int(len(g) * param))
            selected = g.head(n).copy()
            selected["weight"] = selected["pred_score"] / selected["pred_score"].sum()
            port_ret = (selected["fwd_ret_3m"] * selected["weight"]).sum()
        portfolio_rets.append(port_ret)
        dates_used.append(date)
    return pd.Series(portfolio_rets, index=pd.to_datetime(dates_used))

variants = {
    "Top 10% EW": build_portfolio_returns(rebalance_df, "top_pct", 0.10),
    "Top 20% EW": build_portfolio_returns(rebalance_df, "top_pct", 0.20),
    "Top 10 funds": build_portfolio_returns(rebalance_df, "top_n", 10),
    "Top 20 funds": build_portfolio_returns(rebalance_df, "top_n", 20),
    "Score-weighted (top 20%)": build_portfolio_returns(rebalance_df, "score_weighted", 0.20),
}

# benchmark: equal-weighted average of ALL eligible funds each rebalance (passive comparison)
benchmark_ret = rebalance_df.groupby("date")["fwd_ret_3m"].mean()
benchmark_ret.index = pd.to_datetime(benchmark_ret.index)

summary_rows = []
for name, rets in variants.items():
    total_growth = (1 + rets).prod()
    n_periods = len(rets)
    ann_ret = total_growth ** (4 / n_periods) - 1  # 4 periods/year since 3-month rebalance
    ann_vol = rets.std() * np.sqrt(4)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    summary_rows.append({"variant": name, "total_growth": total_growth, "ann_return": ann_ret,
                          "ann_vol": ann_vol, "sharpe": sharpe, "n_periods": n_periods})

bench_growth = (1 + benchmark_ret).prod()
bench_ann_ret = bench_growth ** (4/len(benchmark_ret)) - 1
bench_ann_vol = benchmark_ret.std() * np.sqrt(4)
summary_rows.append({"variant": "Benchmark (equal-weight all eligible)", "total_growth": bench_growth,
                      "ann_return": bench_ann_ret, "ann_vol": bench_ann_vol,
                      "sharpe": bench_ann_ret/bench_ann_vol, "n_periods": len(benchmark_ret)})

portfolio_summary = pd.DataFrame(summary_rows)
print(portfolio_summary.to_string(index=False))

Total test months: 159
Rebalance dates (every 3rd): 53
Rebalance rows: 11,592
                              variant  total_growth  ann_return  ann_vol   sharpe  n_periods
                           Top 10% EW      8.215969    0.172279 0.143225 1.202851         53
                           Top 20% EW      5.680121    0.140073 0.126086 1.110935         53
                         Top 10 funds     11.385087    0.201499 0.199842 1.008290         53
                         Top 20 funds      8.700089    0.177355 0.165284 1.073034         53
             Score-weighted (top 20%)      5.991309    0.144672 0.128957 1.121863         53
Benchmark (equal-weight all eligible)      3.004523    0.086572 0.109302 0.792043         53


In [5]:
funds_per_month = oos_df.groupby("date")["ticker"].nunique()
print("Funds available per rebalance month over time:")
print(funds_per_month.describe())

top10_picks = rebalance_df.sort_values("pred_score", ascending=False).groupby("date").apply(
    lambda g: g.head(max(1, int(len(g) * 0.10)))
)
pick_counts = top10_picks["ticker"].value_counts()
print(f"\nUnique tickers ever picked in Top 10%: {len(pick_counts)}")
print(f"\nMost frequently picked funds:\n{pick_counts.head(15)}")

Funds available per rebalance month over time:
count    159.000000
mean     221.446541
std      147.510251
min       39.000000
25%       84.500000
50%      170.000000
75%      334.500000
max      542.000000
Name: ticker, dtype: float64

Unique tickers ever picked in Top 10%: 437

Most frequently picked funds:
ticker
XS2D    16
DL2P    14
BNKS    13
SPOG    10
SXLY    10
XLDX     9
BCHS     9
SMGB     9
DAGB     9
IGUS     8
IUSE     8
BKCG     8
SWDA     7
XMWD     7
RTYS     7
Name: count, dtype: int64


In [6]:
# estimate turnover: what fraction of holdings change between consecutive rebalances?
def compute_turnover(df, method, param):
    holdings_by_date = {}
    for date, g in df.groupby("date"):
        g = g.sort_values("pred_score", ascending=False)
        if method == "top_pct":
            n = max(1, int(len(g) * param))
        else:
            n = param
        holdings_by_date[date] = set(g.head(n)["ticker"])

    dates_sorted = sorted(holdings_by_date.keys())
    turnovers = []
    for i in range(1, len(dates_sorted)):
        prev_holdings = holdings_by_date[dates_sorted[i-1]]
        curr_holdings = holdings_by_date[dates_sorted[i]]
        changed = len(prev_holdings - curr_holdings)
        turnover = changed / len(prev_holdings) if len(prev_holdings) > 0 else 0
        turnovers.append(turnover)
    return np.mean(turnovers)

for name, method, param in [("Top 10% EW","top_pct",0.10), ("Top 20% EW","top_pct",0.20),
                              ("Top 10 funds","top_n",10), ("Top 20 funds","top_n",20)]:
    t = compute_turnover(rebalance_df, method, param)
    print(f"{name}: avg turnover per rebalance = {t:.1%}")

Top 10% EW: avg turnover per rebalance = 78.3%
Top 20% EW: avg turnover per rebalance = 69.7%
Top 10 funds: avg turnover per rebalance = 76.7%
Top 20 funds: avg turnover per rebalance = 72.9%


In [7]:
COST_SCENARIOS = {
    "Low (15bps round-trip)": 0.0015,
    "Base (30bps round-trip)": 0.0030,
    "High (60bps round-trip)": 0.0060,
}

def apply_costs(gross_rets, turnover_series, cost_per_round_trip):
    # cost drag each period = turnover fraction * round-trip cost
    # (only the changed portion of the portfolio incurs trading cost)
    net_rets = gross_rets.copy()
    for i in range(1, len(net_rets)):
        net_rets.iloc[i] = gross_rets.iloc[i] - (turnover_series[i-1] * cost_per_round_trip)
    return net_rets

def turnover_series_for(df, method, param):
    holdings_by_date = {}
    for date, g in df.groupby("date"):
        g = g.sort_values("pred_score", ascending=False)
        n = max(1, int(len(g)*param)) if method=="top_pct" else param
        holdings_by_date[date] = set(g.head(n)["ticker"])
    dates_sorted = sorted(holdings_by_date.keys())
    turnovers = [0]  # no turnover cost on the very first period (no prior holdings)
    for i in range(1, len(dates_sorted)):
        prev, curr = holdings_by_date[dates_sorted[i-1]], holdings_by_date[dates_sorted[i]]
        turnovers.append(len(prev - curr) / len(prev) if len(prev) > 0 else 0)
    return turnovers

net_results = []
for name, method, param in [("Top 10% EW","top_pct",0.10), ("Top 20% EW","top_pct",0.20),
                              ("Top 10 funds","top_n",10), ("Top 20 funds","top_n",20)]:
    gross = variants[name]
    turnovers = turnover_series_for(rebalance_df, method, param)
    row = {"variant": name, "gross_ann_return": portfolio_summary.set_index("variant").loc[name,"ann_return"],
           "gross_sharpe": portfolio_summary.set_index("variant").loc[name,"sharpe"]}
    for scenario_name, cost in COST_SCENARIOS.items():
        net = apply_costs(gross, turnovers, cost)
        net_growth = (1+net).prod()
        net_ann_ret = net_growth ** (4/len(net)) - 1
        net_ann_vol = net.std() * np.sqrt(4)
        net_sharpe = net_ann_ret / net_ann_vol if net_ann_vol > 0 else np.nan
        row[f"net_ret_{scenario_name}"] = net_ann_ret
        row[f"net_sharpe_{scenario_name}"] = net_sharpe
    net_results.append(row)

net_df = pd.DataFrame(net_results)
pd.set_option('display.width', 150)
print(net_df.to_string(index=False))

     variant  gross_ann_return  gross_sharpe  net_ret_Low (15bps round-trip)  net_sharpe_Low (15bps round-trip)  net_ret_Base (30bps round-trip)  net_sharpe_Base (30bps round-trip)  net_ret_High (60bps round-trip)  net_sharpe_High (60bps round-trip)
  Top 10% EW          0.172279      1.202851                        0.167179                           1.167642                         0.162096                            1.132499                         0.151977                            1.062419
  Top 20% EW          0.140073      1.110935                        0.135650                           1.076881                         0.131240                            1.042836                         0.122457                            0.974783
Top 10 funds          0.201499      1.008290                        0.196423                           0.983287                         0.191362                            0.958328                         0.181286                            0.908549


In [8]:
(ROOT / "results/tables").mkdir(parents=True, exist_ok=True)
net_df.to_csv(ROOT / "results/tables/portfolio_gross_net_comparison.csv", index=False)

portfolio_summary.to_csv(ROOT / "results/tables/portfolio_gross_summary.csv", index=False)

import json
notebook05_summary = {
    "best_variant": "Top 10% EW",
    "gross_sharpe": 1.203,
    "net_sharpe_base_case": 1.132,
    "net_sharpe_high_cost": 1.062,
    "rebalance_frequency": "quarterly (matches 3-month prediction horizon)",
    "avg_turnover_per_rebalance": "78.3%",
    "benchmark_sharpe": 0.792,
    "key_finding": "Top 10% equal-weighted long-only portfolio, built on RF+LightGBM ensemble ranking, outperforms the passive equal-weighted eligible universe at every cost assumption tested (15/30/60bps round-trip), with net Sharpe remaining above 1.0 even in the high-cost scenario.",
    "caveats": [
        "High turnover (~78% per quarter, ~300%+ annualized) means results are sensitive to cost assumptions, though strategy survives all tested scenarios",
        "Top-10%-pick concentration check: 437 unique tickers picked across 53 periods, no evidence of survivorship-driven concentration in a small fund set",
        "Universe size varies significantly over time (39 to 542 funds per month) — early-period portfolios formed from a much smaller eligible set",
    ]
}
with open(ROOT / "results/tables/notebook05_summary.json", "w") as fjson:
    json.dump(notebook05_summary, fjson, indent=2)

print("Notebook 05 core results saved.")

Notebook 05 core results saved.


In [9]:
# build regime labels per rebalance date, using macro_df (already loaded)
regime_df = macro_df[["date", "vix", "uk_gilt_10y"]].copy()
regime_df["date"] = pd.to_datetime(regime_df["date"])

vix_median = regime_df["vix"].median()
regime_df["vol_regime"] = np.where(regime_df["vix"] > vix_median, "High Vol", "Low Vol")

regime_df["gilt_6m_change"] = regime_df["uk_gilt_10y"].diff(6)
regime_df["rate_regime"] = np.where(regime_df["gilt_6m_change"] > 0, "Rising Rates", "Falling Rates")

regime_df["covid_period"] = regime_df["date"].between("2020-02-01", "2020-12-31")

print(f"VIX median used as threshold: {vix_median:.1f}")
print(regime_df[["date","vol_regime","rate_regime","covid_period"]].tail(15))

VIX median used as threshold: 17.4
          date vol_regime    rate_regime  covid_period
211 2025-08-31    Low Vol   Rising Rates         False
212 2025-09-30    Low Vol   Rising Rates         False
213 2025-10-31    Low Vol  Falling Rates         False
214 2025-11-30    Low Vol  Falling Rates         False
215 2025-12-31    Low Vol  Falling Rates         False
216 2026-01-31    Low Vol  Falling Rates         False
217 2026-02-28   High Vol  Falling Rates         False
218 2026-03-31   High Vol   Rising Rates         False
219 2026-04-30    Low Vol   Rising Rates         False
220 2026-05-31    Low Vol   Rising Rates         False
221 2026-06-30    Low Vol   Rising Rates         False
222 2026-07-31    Low Vol   Rising Rates         False
223 2026-08-31    Low Vol   Rising Rates         False
224 2026-09-30    Low Vol  Falling Rates         False
225 2026-10-31    Low Vol  Falling Rates         False


In [10]:
# build the Top 10% EW return series with dates, merge regime labels
top10_rets = variants["Top 10% EW"].reset_index()
top10_rets.columns = ["date", "port_ret"]
top10_rets["date"] = pd.to_datetime(top10_rets["date"])

bench_rets_df = benchmark_ret.reset_index()
bench_rets_df.columns = ["date", "bench_ret"]

regime_perf = top10_rets.merge(regime_df[["date","vol_regime","rate_regime","covid_period"]], on="date", how="left")
regime_perf = regime_perf.merge(bench_rets_df, on="date", how="left")

print("=== Performance by volatility regime ===")
print(regime_perf.groupby("vol_regime").agg(
    mean_port_ret=("port_ret","mean"), mean_bench_ret=("bench_ret","mean"),
    n_periods=("port_ret","count")
))

print("\n=== Performance by rate regime ===")
print(regime_perf.groupby("rate_regime").agg(
    mean_port_ret=("port_ret","mean"), mean_bench_ret=("bench_ret","mean"),
    n_periods=("port_ret","count")
))

print("\n=== COVID period vs rest ===")
print(regime_perf.groupby("covid_period").agg(
    mean_port_ret=("port_ret","mean"), mean_bench_ret=("bench_ret","mean"),
    n_periods=("port_ret","count")
))

print("\n=== The two outlier periods specifically ===")
print(regime_perf[regime_perf["date"].between("2022-06-01","2023-12-31")][
    ["date","port_ret","bench_ret","vol_regime","rate_regime"]])

=== Performance by volatility regime ===
            mean_port_ret  mean_bench_ret  n_periods
vol_regime                                          
High Vol         0.073310        0.041092         19
Low Vol          0.025892        0.011990         34

=== Performance by rate regime ===
               mean_port_ret  mean_bench_ret  n_periods
rate_regime                                            
Falling Rates       0.054062        0.034978         25
Rising Rates        0.032917        0.011213         28

=== COVID period vs rest ===
              mean_port_ret  mean_bench_ret  n_periods
covid_period                                          
False              0.037351        0.018422         50
True               0.135233        0.089107          3

=== The two outlier periods specifically ===
         date  port_ret  bench_ret vol_regime    rate_regime
37 2022-07-31  0.039164  -0.017394   High Vol   Rising Rates
38 2022-10-31  0.079337   0.065578   High Vol   Rising Rates
39 2023-

In [11]:
regime_summary = {
    "vol_regime": regime_perf.groupby("vol_regime")[["port_ret","bench_ret"]].mean().to_dict(),
    "rate_regime": regime_perf.groupby("rate_regime")[["port_ret","bench_ret"]].mean().to_dict(),
    "covid_period": regime_perf.groupby("covid_period")[["port_ret","bench_ret"]].mean().to_dict(),
    "key_finding": "Strategy outperforms passive benchmark across every tested regime (high/low vol, rising/falling rates), with no regime where the edge disappears. COVID-period result (n=3) is directionally consistent but too small a sample for statistical confidence.",
}
import json
with open(ROOT / "results/tables/regime_analysis_summary.json", "w") as fjson:
    json.dump(regime_summary, fjson, indent=2, default=str)

regime_perf.to_csv(ROOT / "results/tables/regime_performance_detail.csv", index=False)
print("Regime analysis saved.")

Regime analysis saved.


In [14]:
def run_ensemble_ic_for_window(train_months, test_months=3):
    local_folds = []
    i = train_months
    while i + test_months <= len(all_dates):
        train_dates = all_dates[i-train_months:i]
        test_dates = all_dates[i:i+test_months]
        local_folds.append({"train_start": train_dates[0], "train_end": train_dates[-1],
                             "test_start": test_dates[0], "test_end": test_dates[-1]})
        i += test_months

    fold_ics = []
    for f in local_folds:
        train_mask = (data_v3["date"] >= f["train_start"]) & (data_v3["date"] <= f["train_end"])
        test_mask = (data_v3["date"] >= f["test_start"]) & (data_v3["date"] <= f["test_end"])
        train = data_v3[train_mask]
        test = data_v3[test_mask]
        if len(train) < 50 or len(test) < 5:
            continue
        X_train_raw = train[FEATURE_COLS_V2]
        y_train = train["xs_excess_ret_3m"]
        X_test_raw = test[FEATURE_COLS_V2]
        X_train_filled = X_train_raw.fillna(0)
        X_test_filled = X_test_raw.fillna(0)

        lgb_m = lgb.LGBMRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                                   min_child_samples=20, random_state=42, verbose=-1)
        lgb_m.fit(X_train_raw, y_train)
        lgb_preds = lgb_m.predict(X_test_raw)

        rf_m = RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=20,
                                      random_state=42, n_jobs=-1)
        rf_m.fit(X_train_filled, y_train)
        rf_preds = rf_m.predict(X_test_filled)

        pred_df = pd.DataFrame()
        pred_df["LightGBM"] = lgb_preds
        pred_df["RF"] = rf_preds
        blended = pred_df.rank(pct=True).mean(axis=1)

        test_copy = test.copy()
        test_copy["pred"] = blended.values
        monthly_ic = test_copy.groupby("date").apply(
            lambda g: spearmanr(g["pred"], g["xs_excess_ret_3m"]).correlation if len(g) > 5 else np.nan)
        fold_ics.append(monthly_ic.mean())
    return pd.Series(fold_ics).dropna(), len(local_folds)

print("Function defined OK")

Function defined OK


In [16]:
import statsmodels.api as sm

In [17]:
window_results = {}
for window in [24, 36, 60]:
    ics, n_folds = run_ensemble_ic_for_window(window)
    nw = sm.OLS(ics.values, sm.add_constant(np.ones(len(ics)))).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
    window_results[window] = {"mean_ic": ics.mean(), "n_folds": n_folds, "nw_t": nw.tvalues[0], "nw_p": nw.pvalues[0]}
    print(f"Train window={window}mo: mean IC={ics.mean():.4f}, folds={n_folds}, NW t={nw.tvalues[0]:.2f}, NW p={nw.pvalues[0]:.4f}")

Train window=24mo: mean IC=0.0458, folds=57, NW t=2.25, NW p=0.0242
Train window=36mo: mean IC=0.0499, folds=53, NW t=3.25, NW p=0.0012
Train window=60mo: mean IC=0.0380, folds=45, NW t=2.23, NW p=0.0259


In [18]:
robustness_summary = {"train_window_months": window_results}

(ROOT / "results/tables").mkdir(parents=True, exist_ok=True)
with open(ROOT / "results/tables/robustness_train_window.json", "w") as fjson:
    json.dump(robustness_summary, fjson, indent=2, default=str)
print("Train window robustness saved.")

Train window robustness saved.


In [19]:
# portfolio size robustness — using the already-built oos_df predictions (36mo window, our primary config)
# so this reuses rebalance_df / variants logic from earlier, just testing more percentile cuts
size_variants = {}
for pct in [0.05, 0.10, 0.15, 0.20, 0.30]:
    rets = build_portfolio_returns(rebalance_df, "top_pct", pct)
    ann_ret = (1+rets).prod() ** (4/len(rets)) - 1
    ann_vol = rets.std() * np.sqrt(4)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    size_variants[f"Top {int(pct*100)}%"] = {"ann_return": ann_ret, "ann_vol": ann_vol, "sharpe": sharpe}
    print(f"Top {int(pct*100)}%: ann_return={ann_ret:.4f}, sharpe={sharpe:.3f}")

Top 5%: ann_return=0.2062, sharpe=1.110
Top 10%: ann_return=0.1723, sharpe=1.203
Top 15%: ann_return=0.1532, sharpe=1.159
Top 20%: ann_return=0.1401, sharpe=1.111
Top 30%: ann_return=0.1267, sharpe=1.076


In [20]:
with open(ROOT / "results/tables/robustness_portfolio_size.json", "w") as fjson:
    json.dump(size_variants, fjson, indent=2, default=str)
print("Portfolio size robustness saved.")

Portfolio size robustness saved.


In [21]:
# monthly overlapping rebalance: use ALL test months (not just every 3rd), each holding for 3m
# this creates overlapping positions - need average return across 3 concurrently-held "tranches"
monthly_rebal_df = oos_df.copy()  # already has pred_score for every test month, not just every 3rd

monthly_top10_rets = build_portfolio_returns(monthly_rebal_df, "top_pct", 0.10)

# since positions overlap (each held 3mo, formed every 1mo), the realized monthly portfolio
# return is the average of the 3 "tranches" currently active - approximate via rolling mean
monthly_top10_rets_sorted = monthly_top10_rets.sort_index()
blended_monthly_ret = monthly_top10_rets_sorted.rolling(3, min_periods=1).mean()

ann_ret_monthly = (1 + blended_monthly_ret).prod() ** (12/len(blended_monthly_ret)) - 1
ann_vol_monthly = blended_monthly_ret.std() * np.sqrt(12)
sharpe_monthly = ann_ret_monthly / ann_vol_monthly

print(f"Monthly overlapping rebalance, Top 10%: ann_return={ann_ret_monthly:.4f}, "
      f"ann_vol={ann_vol_monthly:.4f}, sharpe={sharpe_monthly:.3f}")
print(f"(vs quarterly non-overlapping: ann_return=0.1723, sharpe=1.203)")

Monthly overlapping rebalance, Top 10%: ann_return=0.4519, ann_vol=0.1865, sharpe=2.422
(vs quarterly non-overlapping: ann_return=0.1723, sharpe=1.203)


In [22]:
# discard the flawed monthly-overlapping number; document the limitation instead
robustness_notes = {
    "train_window": "Tested 24/36/60mo — signal robust across all (p<0.03 each), 36mo strongest (p=0.0012)",
    "portfolio_size": "Sharpe peaks cleanly at Top 10% (1.203), smooth decline on both sides — not an isolated spike",
    "rebalance_frequency": "Only quarterly (matching 3m prediction horizon) rigorously tested. Monthly overlapping rebalancing was attempted but requires tranche-level single-month return accounting not yet implemented; deferred as a documented limitation rather than reporting an unreliable figure.",
}
with open(ROOT / "results/tables/robustness_notes.json", "w") as fjson:
    json.dump(robustness_notes, fjson, indent=2)
print("Robustness notes saved.")

Robustness notes saved.


In [23]:
portfolio_returns_export = pd.DataFrame({
    "date": variants["Top 10% EW"].index,
    "top10_ew_ret": variants["Top 10% EW"].values,
    "benchmark_ret": benchmark_ret.reindex(variants["Top 10% EW"].index).values,
})
portfolio_returns_export.to_csv(ROOT / "results/tables/portfolio_return_series.csv", index=False)
print(f"Saved {len(portfolio_returns_export)} rows")
portfolio_returns_export.head()

Saved 53 rows


,date,top10_ew_ret,benchmark_ret
0,2013-04-30,0.060569,-0.005710
1,2013-07-31,0.032829,0.040825
2,2013-10-31,0.007192,-0.048048
3,2014-01-31,0.011711,0.049348
4,2014-04-30,0.024828,0.020212
